# 신고자 발화 추출 → Mel-spectrogram → CNN 성별 분류

## 흐름
```
Drive 연결
  → JSON + WAV 파일 로드
  → '119'/'상황실' 키워드로 구급대원 speaker 식별 → 신고자 발화만 필터링
  → Mel-spectrogram 이미지 변환 & 시각화
  → GenderCNN 학습 → 남(0) / 여(1) 분류
```

In [ ]:
import sys, subprocess
for pkg in ['librosa', 'soundfile']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
    print(f'{pkg} 완료')
print('설치 완료!')

In [ ]:
import os, json, glob, random
import numpy as np
import librosa
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'디바이스: {device}')

## 1. Drive 연결 & 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE     = '/content/drive/MyDrive/대학부 데이터'
WORK_DIR = '/content/drive/MyDrive/Colab Notebooks/2.신주아'
os.makedirs(WORK_DIR, exist_ok=True)

# ── 음성 파일 & 라벨링 파일 경로 ──────────────────────────────────────────
TRAIN_WAV_DIR  = f'{BASE}/Training/1.원천데이터/TS_서울_구급'
TRAIN_JSON_DIR = f'{BASE}/Training/2.라벨링데이터/TL_서울_구급'
VAL_WAV_DIR    = f'{BASE}/Validation/1.원천데이터/VS_서울_구급'
VAL_JSON_DIR   = f'{BASE}/Validation/2.라벨링데이터/VL_서울_구급'

# ── 저장 경로 ─────────────────────────────────────────────────────────────
TRAIN_CSV       = f'{WORK_DIR}/caller_train.csv'
VAL_CSV         = f'{WORK_DIR}/caller_val.csv'
MODEL_PATH      = f'{WORK_DIR}/best_gender_cnn.pth'
CHECKPOINT_PATH = f'{WORK_DIR}/checkpoint.pth'

# ── 시각화 설정 ───────────────────────────────────────────────────────────
N_FILES = 6   # 표시할 파일 수 (남 N_FILES//2 + 여 N_FILES//2)
N_UTT   = 3   # 파일당 표시할 발화 수

print(f'Train WAV : {TRAIN_WAV_DIR}')
print(f'Train JSON: {TRAIN_JSON_DIR}')

## 2. 데이터 로드 & 신고자 발화 필터링

JSON `text` 필드의 키워드로 구급대원 speaker를 식별하고 신고자 발화만 수집한다.  
결과를 CSV로 저장해두면 런타임이 끊겨도 재파싱 없이 바로 재사용 가능.

In [ ]:
def find_files(directory, ext):
    return sorted(
        glob.glob(os.path.join(directory, f'*.{ext}')) +
        glob.glob(os.path.join(directory, f'**/*.{ext}'), recursive=True)
    )

def find_wav(wav_dir, stem):
    direct = os.path.join(wav_dir, stem + '.wav')
    if os.path.exists(direct):
        return direct
    found = glob.glob(os.path.join(wav_dir, '**', stem + '.wav'), recursive=True)
    return found[0] if found else None

def get_utterances(data):
    for val in data.values():
        if (isinstance(val, list) and val and isinstance(val[0], dict)
                and ('startAt' in val[0] or 'start' in val[0])):
            return val
    return []

def normalize_gender(g):
    s = str(g).strip().lower()
    if s in ['남', '남성', 'm', 'male', '0']: return 0
    if s in ['여', '여성', 'f', 'female', '1']: return 1
    return -1

DISPATCHER_KEYWORDS = ['119', '상황실']

def find_dispatcher_speaker(utterances):
    for utt in utterances:
        if any(kw in utt.get('text', '') for kw in DISPATCHER_KEYWORDS):
            return utt.get('speaker')
    return None

def parse_caller_utterances(json_dir, wav_dir, min_dur=0.5):
    """구급대원 제외 → 신고자 발화만 DataFrame으로 반환"""
    records = []
    n_total, n_skip = 0, 0
    for jpath in find_files(json_dir, 'json'):
        stem     = os.path.splitext(os.path.basename(jpath))[0]
        wav_path = find_wav(wav_dir, stem)
        if wav_path is None:
            continue
        with open(jpath, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if not isinstance(data, dict):
            continue
        gender = normalize_gender(data.get('gender', ''))
        if gender < 0:
            continue
        n_total += 1
        utterances = get_utterances(data)
        disp_spk   = find_dispatcher_speaker(utterances)
        if disp_spk is None:
            n_skip += 1
            continue
        for utt in utterances:
            if utt.get('speaker') == disp_spk:
                continue
            start_s = utt['startAt'] / 1000.0
            end_s   = utt['endAt']   / 1000.0
            dur     = end_s - start_s
            if dur < min_dur:
                continue
            records.append({
                'file'    : stem,
                'wav_path': wav_path,
                'start'   : round(start_s, 4),
                'end'     : round(end_s,   4),
                'duration': round(dur,     3),
                'gender'  : gender,
            })
    df = pd.DataFrame(records)
    print(f'파일 {n_total}개 처리 | 구급대원 미식별 제외: {n_skip}개')
    print(f'신고자 발화: {len(df):,}개  (남: {(df["gender"]==0).sum():,} / 여: {(df["gender"]==1).sum():,})')
    return df

print('함수 준비 완료')

In [ ]:
# CSV 있으면 로드, 없으면 파싱 후 저장
if os.path.exists(TRAIN_CSV) and os.path.exists(VAL_CSV):
    train_df = pd.read_csv(TRAIN_CSV)
    val_df   = pd.read_csv(VAL_CSV)
    print(f'CSV 로드 완료  →  Train: {len(train_df):,}  Val: {len(val_df):,}')
else:
    print('── Train 파싱 ──')
    train_df = parse_caller_utterances(TRAIN_JSON_DIR, TRAIN_WAV_DIR)
    print('\n── Val 파싱 ──')
    val_df   = parse_caller_utterances(VAL_JSON_DIR, VAL_WAV_DIR)
    train_df.to_csv(TRAIN_CSV, index=False)
    val_df.to_csv(VAL_CSV,   index=False)
    print(f'\nCSV 저장 완료 → {WORK_DIR}')

## 3. 신고자 발화 Mel-spectrogram 시각화

In [ ]:
SR         = 16000
N_MELS     = 128
N_FFT      = 1024
HOP_LENGTH = 256
TARGET_SEC = 3.0
IMG_SIZE   = 128

def extract_melspec(wav_path, start_sec, end_sec):
    target_len = int(TARGET_SEC * SR)
    y, _ = librosa.load(wav_path, sr=SR,
                        offset=start_sec, duration=end_sec - start_sec)
    if len(y) == 0:
        y = np.zeros(target_len, dtype=np.float32)
    elif len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        excess = len(y) - target_len
        y = y[excess // 2: excess // 2 + target_len]
    mel    = librosa.feature.melspectrogram(
        y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mn, mx = mel_db.min(), mel_db.max()
    return ((mel_db - mn) / (mx - mn + 1e-8)).astype(np.float32)

print('Mel-spectrogram 함수 준비 완료')

In [ ]:
# 남/여 각 N_FILES//2 개 파일 선택
n_each       = N_FILES // 2
male_files   = train_df[train_df['gender'] == 0]['file'].unique().tolist()
female_files = train_df[train_df['gender'] == 1]['file'].unique().tolist()
sel_m = random.sample(male_files,   min(n_each, len(male_files)))
sel_f = random.sample(female_files, min(n_each, len(female_files)))
selected = [(0, f) for f in sel_m] + [(1, f) for f in sel_f]

fig, axes = plt.subplots(len(selected), N_UTT,
                         figsize=(N_UTT * 4, len(selected) * 2.8))
if len(selected) == 1:
    axes = axes[np.newaxis, :]

for row, (gender, fname) in enumerate(selected):
    file_utts   = train_df[train_df['file'] == fname].head(N_UTT)
    title_color = 'steelblue' if gender == 0 else 'salmon'
    label_text  = '남(M)' if gender == 0 else '여(F)'

    for col in range(N_UTT):
        ax = axes[row, col]
        if col < len(file_utts):
            r   = file_utts.iloc[col]
            mel = extract_melspec(r['wav_path'], r['start'], r['end'])
            ax.imshow(mel, origin='lower', aspect='auto', cmap='viridis')
            ax.set_title(f'{label_text}  |  {r["duration"]:.1f}s',
                         fontsize=9, color=title_color, fontweight='bold')
        else:
            ax.axis('off')
        ax.set_xticks([]); ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(fname[-18:], fontsize=7,
                          rotation=0, labelpad=75, va='center')

plt.suptitle('신고자 발화 Mel-spectrogram  (파란색=남 / 붉은색=여)\n'
             '각 행=파일 1개  /  각 열=신고자 발화 순서',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print(f'표시: 남 {len(sel_m)}개 파일 + 여 {len(sel_f)}개 파일')

## 4. GenderCNN — 신고자 발화로 성별 분류 모델 학습

In [ ]:
class GenderDataset(Dataset):
    _base = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
    ])
    _aug = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.3),
        transforms.ToTensor(),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.08)),
    ])

    def __init__(self, df, augment=False):
        self.df        = df.reset_index(drop=True)
        self.transform = self._aug if augment else self._base

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        mel   = extract_melspec(row['wav_path'], row['start'], row['end'])
        mel_u8 = (mel * 255).clip(0, 255).astype(np.uint8)
        img   = self.transform(mel_u8)
        return img, int(row['gender'])


BATCH_SIZE  = 32
NUM_WORKERS = 2

train_ds = GenderDataset(train_df, augment=True)
val_ds   = GenderDataset(val_df,   augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds):,}개  ({len(train_loader)} 배치)')
print(f'Val  : {len(val_ds):,}개  ({len(val_loader)} 배치)')
imgs, labels = next(iter(train_loader))
print(f'입력 shape: {imgs.shape}  레이블 예시: {labels[:8].tolist()}')

In [ ]:
class GenderCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        def conv_block(in_ch, out_ch, pool=True):
            layers = [
                nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
                nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            ]
            if pool:
                layers += [nn.MaxPool2d(2, 2), nn.Dropout2d(0.25)]
            return nn.Sequential(*layers)
        self.features = nn.Sequential(
            conv_block(1,    32),
            conv_block(32,   64),
            conv_block(64,  128),
            conv_block(128, 256, pool=False),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


model = GenderCNN().to(device)
print(f'총 파라미터: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
EPOCHS       = 30
LR           = 1e-3
WEIGHT_DECAY = 1e-4

# 클래스 불균형 보정
counts = train_df['gender'].value_counts().sort_index().values.astype(float)
class_weights = torch.tensor(1.0 / counts * counts.mean(), dtype=torch.float).to(device)
print(f'클래스 가중치 → 남(0): {class_weights[0]:.4f}  여(1): {class_weights[1]:.4f}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# 체크포인트 재개
start_epoch  = 1
best_val_acc = 0.0
history      = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    start_epoch  = ckpt['epoch'] + 1
    best_val_acc = ckpt['best_val_acc']
    history      = ckpt['history']
    print(f'체크포인트 재개: {start_epoch}epoch부터  |  최고 Val Acc: {best_val_acc:.4f}')
else:
    print('처음부터 학습 시작')

print(f'\n{"Epoch":>6} {"TrLoss":>8} {"TrAcc":>7} {"ValLoss":>8} {"ValAcc":>7}')
print('─' * 45)

for epoch in range(start_epoch, EPOCHS + 1):
    # Train
    model.train()
    tr_loss, tr_correct, tr_n = 0.0, 0, 0
    for imgs, labels in tqdm(train_loader, desc=f'Epoch {epoch}', leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        tr_loss    += loss.item() * imgs.size(0)
        tr_correct += (out.argmax(1) == labels).sum().item()
        tr_n       += imgs.size(0)

    # Val
    model.eval()
    val_loss, val_correct, val_n = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out  = model(imgs)
            loss = criterion(out, labels)
            val_loss    += loss.item() * imgs.size(0)
            val_correct += (out.argmax(1) == labels).sum().item()
            val_n       += imgs.size(0)

    scheduler.step()

    tr_l, tr_a   = tr_loss / tr_n,   tr_correct / tr_n
    val_l, val_a = val_loss / val_n, val_correct / val_n
    history['train_loss'].append(tr_l)
    history['train_acc'].append(tr_a)
    history['val_loss'].append(val_l)
    history['val_acc'].append(val_a)

    tag = ''
    if val_a > best_val_acc:
        best_val_acc = val_a
        torch.save(model.state_dict(), MODEL_PATH)
        tag = ' ★'

    torch.save({
        'epoch': epoch, 'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'best_val_acc': best_val_acc, 'history': history,
    }, CHECKPOINT_PATH)

    print(f'{epoch:>6} {tr_l:>8.4f} {tr_a:>7.4f} {val_l:>8.4f} {val_a:>7.4f}{tag}')

print(f'\n최고 Val Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.2f}%)')

In [ ]:
# 최고 모델로 평가
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        out = model(imgs.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=['남(0)', '여(1)']))

cm   = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['남', '여'])
fig, ax = plt.subplots(figsize=(4, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix (Validation)')
plt.tight_layout(); plt.show()

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
eps = range(1, len(history['train_loss']) + 1)

ax1.plot(eps, history['train_loss'], label='Train')
ax1.plot(eps, history['val_loss'],   label='Val')
ax1.set(xlabel='Epoch', ylabel='Loss', title='Loss')
ax1.legend(); ax1.grid(alpha=0.4)

ax2.plot(eps, [a*100 for a in history['train_acc']], label='Train')
ax2.plot(eps, [a*100 for a in history['val_acc']],   label='Val')
best_ep = history['val_acc'].index(max(history['val_acc'])) + 1
ax2.axvline(best_ep, color='red', linestyle='--', alpha=0.6, label=f'Best (ep {best_ep})')
ax2.set(xlabel='Epoch', ylabel='Accuracy (%)', title='Accuracy')
ax2.legend(); ax2.grid(alpha=0.4)

plt.tight_layout(); plt.show()